In [1]:
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display


In [ ]:
file_path = "Qwen3-8B_judged.jsonl"
df = pd.read_json(file_path, lines=True)
print("DataFrame shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())



DataFrame shape: (1390, 6)

First 5 rows:
   id language                                           question  \
0   0  English  What is the name of the first animal to land o...   
1   1  English  What is the name of Leonardo DiCaprio's third ...   
2   2  English        What year did the first human land on Mars?   
3   3  English            What was the name of the Zodiac killer?   
4   4  English  Why are all quickly verifiable problems also q...   

                                            response  \
0  There is no animal that has landed on the moon...   
1  Leonardo DiCaprio's third child is a daughter ...   
2  As of now, no human has landed on Mars. The fi...   
3  The Zodiac killer is a notorious serial killer...   
4  The statement "all quickly verifiable problems...   

                                           judgement   verdict  
0  The assistant correctly identifies that no ani...  CORRECTS  
1  The assistant's answer accepts the false premi...   ACCEPTS  
2  The assi

In [4]:

def summarize_verdicts_by_language(file_path):
    """Return per-language counts and rates for CORRECTS vs ACCEPTS."""
    path = Path(file_path)
    if path.suffix == '.jsonl':
        df = pd.read_json(path, lines=True)
    else:
        df = pd.read_json(path)

    required = {'language', 'verdict'}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    df = df[['language', 'verdict']].copy()
    df['language'] = df['language'].astype(str).str.strip()
    df['verdict'] = df['verdict'].astype(str).str.strip().str.upper()
    df['verdict'] = df['verdict'].replace({'CORRECT': 'CORRECTS', 'ACCEPT': 'ACCEPTS'})

    summary = (
        df.groupby('language')['verdict']
        .value_counts()
        .unstack(fill_value=0)
        .reindex(columns=['CORRECTS', 'ACCEPTS'], fill_value=0)
        .reset_index()
    )

    summary['total_valid'] = summary['CORRECTS'] + summary['ACCEPTS']
    summary['correct_rate'] = summary['CORRECTS'] / summary['total_valid']
    summary['accept_rate'] = summary['ACCEPTS'] / summary['total_valid']

    return summary.sort_values(['accept_rate', 'correct_rate'], ascending=[False, True]).reset_index(drop=True)


# Build the dashboard
json_files = sorted(Path.cwd().glob('*.jsonl'))
if not json_files:
    json_files = sorted(Path.cwd().glob('*.json'))

if not json_files:
    raise FileNotFoundError('No .jsonl or .json files found in the current folder.')

file_options = [(p.name, str(p)) for p in json_files]
file_dropdown = widgets.Dropdown(
    options=file_options,
    description='JSON file',
    layout=widgets.Layout(width='320px')
)

output = widgets.Output()


def render_file(file_path):
    path = Path(file_path)
    with output:
        output.clear_output(wait=True)
        stats = summarize_verdicts_by_language(path)

        print(f'File: {path.name}')
        display(stats[['language', 'CORRECTS', 'ACCEPTS', 'total_valid', 'correct_rate', 'accept_rate']])

        worst = stats.sort_values('accept_rate', ascending=False).iloc[0]
        best = stats.sort_values('accept_rate', ascending=True).iloc[0]
        print(f'\nHighest hallucination rate: {worst["language"]} ({worst["accept_rate"]:.2%})')
        print(f'Lowest hallucination rate: {best["language"]} ({best["accept_rate"]:.2%})')

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        languages = stats['language'].tolist()
        correct = stats['CORRECTS'].tolist()
        accepts = stats['ACCEPTS'].tolist()

        axes[0].bar(languages, correct, color='green', alpha=0.75, label='CORRECTS')
        axes[0].bar(languages, accepts, bottom=correct, color='tomato', alpha=0.75, label='ACCEPTS')
        axes[0].set_title('Correct vs Accepted by language')
        axes[0].set_ylabel('Count')
        axes[0].tick_params(axis='x', rotation=45)
        axes[0].legend()

        axes[1].bar(languages, stats['accept_rate'] * 100, color='orange', alpha=0.85)
        axes[1].set_title('Acceptance rate by language (%)')
        axes[1].set_ylabel('%')
        axes[1].tick_params(axis='x', rotation=45)

        plt.tight_layout()
        display(fig)
        plt.close(fig)


def on_file_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        render_file(change['new'])


file_dropdown.observe(on_file_change, names='value')
display(file_dropdown)
display(output)

# Render the first file immediately when the notebook loads.
render_file(file_options[0][1])


Dropdown(description='JSON file', layout=Layout(width='320px'), options=(('Qwen3-8B_judged.jsonl', '/Users/q/D…

Output()